# EKS REST API를 AgentCore Gateway에 연결하기

이 실습에서는 프라이빗 VPC 내부의 Amazon EKS에 REST API(FastAPI)를 배포한 다음, 내부 NLB를 통한 관리형 VPC 송신을 사용하여 [Amazon Bedrock AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html)에 **OpenAPI 스키마를 사용하는 MCP 대상으로** 연결합니다.

VPC와 연결된 Route 53 프라이빗 호스팅 영역의 **프라이빗 도메인**을 통해 API에 접근할 수 있습니다. VPC에서 **Private DNS**가 활성화되어 있으면(기본값), AgentCore Gateway의 관리형 Resource Gateway가 VPC의 DNS 해석기를 통해 도메인을 해석합니다.

MCP 서버 실습과 달리 이 실습에서는 **OpenAPI 스키마**를 사용하여 API 엔드포인트를 설명합니다. AgentCore Gateway는 이 스키마를 사용하여 API 작업을 AI 에이전트가 호출할 수 있는 도구로 노출합니다.

VPC 송신, 인증서 요구 사항 및 Private DNS에 대한 배경 정보는 [프로젝트 README](../README.md), [Managed VPC Resource README 문서](../01-managed-vpc-resource/README.md) 및 [사전 요구 사항](../00-prerequisites/)을 참조하세요.

![아키텍처](./images/eks-api.png)

## 사전 요구 사항

- [실습 0](../00-prerequisites/00-vpc-gateway-setup.ipynb) 완료(VPC + AgentCore Gateway 배포 완료)
- Docker 실행 중(CDK 컨테이너 이미지 빌드용)
- NLB에서 TLS를 종료하기 위한 [ACM 퍼블릭 인증서](../00-prerequisites/create-acm-public-certificate.md)

> **중요:** 이전에 [MCP 서버 Notebook](./mcp-server-gateway-managed.ipynb)을 실행했다면 이 실습을 배포하기 전에 해당 스택을 먼저 정리하세요(Gateway 대상을 삭제하고 `cdk destroy McpEks ...` 실행). Shared EKS Cluster는 삭제하지 않아도 됩니다.

## 1단계: 종속성 설치 및 라이브러리 가져오기

In [ ]:
import os
from pathlib import Path

# 프로젝트 루트로 이동
cwd = Path.cwd()
while cwd != cwd.parent:
    if (cwd / "cdk.json").exists():
        break
    cwd = cwd.parent
os.chdir(cwd)
print(f"Working directory: {os.getcwd()}")

!pip install --force-reinstall -q -r requirements.txt

In [ ]:
import json
import os
import time

import boto3
from utils.utils import get_token

# 실습 0에서 변수 복원
%store -r ACCOUNT_A_ID
%store -r ACCOUNT_A_PROFILE
%store -r GATEWAY_ID
%store -r GATEWAY_URL
%store -r USER_POOL_ID
%store -r USER_POOL_CLIENT_ID
%store -r TOKEN_ENDPOINT_URL
%store -r OAUTH_SCOPES
%store -r VPC_USW2_ID
%store -r VPC_USW2_PRIVATE_SUBNETS

os.environ["ACCOUNT_A_ID"] = ACCOUNT_A_ID

REGION = "us-west-2"
session = boto3.Session(profile_name=ACCOUNT_A_PROFILE, region_name=REGION)
agentcore = session.client("bedrock-agentcore-control")

# Cognito 클라이언트 암호 가져오기
cognito = session.client("cognito-idp")
client_desc = cognito.describe_user_pool_client(UserPoolId=USER_POOL_ID, ClientId=USER_POOL_CLIENT_ID)
CLIENT_SECRET = client_desc["UserPoolClient"]["ClientSecret"]

print(f"Account:    {ACCOUNT_A_ID}")
print(f"Region:     {REGION}")
print(f"Gateway ID: {GATEWAY_ID}")
print(f"VPC ID:     {VPC_USW2_ID}")

In [ ]:
CERT_ARN = input("ACM public certificate ARN: ").strip()
DOMAIN = input("Domain name covered by the certificate (e.g., api.internal.yourcompany.com): ").strip()

assert CERT_ARN.startswith("arn:aws:acm:"), "Invalid certificate ARN"
assert not DOMAIN.startswith("http"), "Domain should not include http:// or https://"
assert "." in DOMAIN, "Domain must contain at least one dot"
assert " " not in DOMAIN, "Domain must not contain whitespace"

print(f"Cert ARN: {CERT_ARN}")
print(f"Domain:   {DOMAIN}")

## 2단계: EKS에 REST API 배포

이 CDK 스택은 다음을 배포합니다.
- 포트 8080에서 Kubernetes Deployment로 실행되는 **REST API**(FastAPI: /health, /items GET/POST)
- ACM 인증서를 사용하는 TLS 리스너(포트 443)가 있는 **내부 NLB**(K8s Service 주석을 통해 생성)
- VPC와 연결되고 이름이 `<DOMAIN>`인 **Route 53 프라이빗 호스팅 영역**(처음에는 비어 있으며, NLB가 프로비저닝되면 Notebook에서 NLB를 가리키는 Alias 레코드를 추가)

> **참고:** 이 실습에서는 MCP 서버 실습 또는 실습 0을 통해 Shared EKS Cluster(AWS Load Balancer Controller 포함)가 이미 배포되어 있다고 가정합니다.

In [ ]:
# # 공유 EKS 클러스터 배포(이미 배포된 경우 건너뛰며, --exclusively는 스택 간 업데이트 문제를 방지)
# !ACCOUNT_A_ID={ACCOUNT_A_ID} cdk deploy SharedEksCluster \
#     --profile {ACCOUNT_A_PROFILE} \
#     --require-approval never \
#     --outputs-file eks-cluster-outputs.json \
#     --exclusively

In [ ]:
# NLB + 프라이빗 호스팅 영역과 함께 EKS에 API 서버 배포
!ACCOUNT_A_ID={ACCOUNT_A_ID} cdk deploy ApiEks \
    -c "publicCertArn={CERT_ARN}" \
    -c "privateDomain={DOMAIN}" \
    --profile {ACCOUNT_A_PROFILE} \
    --require-approval never \
    --outputs-file eks-api-outputs.json

In [ ]:
# CDK 출력 읽기(프라이빗 호스팅 영역은 배포 시 생성되며 NLB DNS는 아래에서 채움)
with open("eks-api-outputs.json") as f:
    eks_api_outputs = json.load(f)["ApiEks"]

PRIVATE_ZONE_ID = eks_api_outputs["PrivateZoneId"]
PRIVATE_DOMAIN = eks_api_outputs["PrivateDomain"]
print(f"Private hosted zone: {PRIVATE_DOMAIN}  (zone ID: {PRIVATE_ZONE_ID})")

# K8s에서 관리하는 NLB 검색
print("\nWaiting for NLB to be provisioned by the AWS Load Balancer Controller...")

elbv2_client = session.client("elbv2")
ec2_client = session.client("ec2")
route53_client = session.client("route53")


def _nlb_has_healthy_target(nlb_arn):
    """이 NLB의 target group 중 하나라도 정상 target을 포함하면 True를 반환합니다.

    이 검사가 필요한 이유: K8s Service를 재배포하거나 stack 반복 사이에서 이동하면
    AWS Load Balancer Controller가 오래된 pod IP를 가리키는 고립된 NLB를 남길 수
    있습니다. 두 NLB가 모두 이름 필터에 일치해도 실제 pod로 라우팅되는 것은 하나뿐입니다.
    우연히 첫 번째 항목을 선택하면 비정상 pod로 라우팅되어 gateway target 호출 시
    "Connection closed by peer"가 표시될 수 있습니다.
    """
    tgs = elbv2_client.describe_target_groups(LoadBalancerArn=nlb_arn)["TargetGroups"]
    for tg in tgs:
        health = elbv2_client.describe_target_health(TargetGroupArn=tg["TargetGroupArn"])["TargetHealthDescriptions"]
        if any(t["TargetHealth"]["State"] == "healthy" for t in health):
            return True
    return False


NLB_DNS = None
NLB_HOSTED_ZONE_ID = None
NLB_SG_ID = None
for attempt in range(20):
    nlbs = elbv2_client.describe_load_balancers()["LoadBalancers"]
    api_nlbs = [
        n
        for n in nlbs
        if n.get("VpcId") == VPC_USW2_ID
        and n["Scheme"] == "internal"
        and n["Type"] == "network"
        and "restapi" in n.get("LoadBalancerName", "").lower().replace("-", "")
    ]
    # 정상 상태 대상이 하나 이상 있는 NLB를 우선하여 고립된 NLB 건너뛰기
    healthy = [n for n in api_nlbs if _nlb_has_healthy_target(n["LoadBalancerArn"])]
    chosen = healthy[0] if healthy else (api_nlbs[0] if api_nlbs else None)
    if chosen and (healthy or attempt >= 5):
        # 정상 상태 NLB가 있거나 충분히 기다린 경우 유일한 후보 선택
        nlb = chosen
        NLB_DNS = nlb["DNSName"]
        NLB_HOSTED_ZONE_ID = nlb["CanonicalHostedZoneId"]
        NLB_SG_ID = nlb["SecurityGroups"][0] if nlb.get("SecurityGroups") else None
        if len(api_nlbs) > 1:
            stale = [n["LoadBalancerName"] for n in api_nlbs if n is not nlb]
            print(
                f"WARNING: Multiple matching NLBs found; chose {nlb['LoadBalancerName']}. "
                f"Stale (orphaned by AWS LB Controller): {stale}"
            )
        break
    print(f"  Waiting... (attempt {attempt + 1}/20)")
    time.sleep(15)

assert NLB_DNS, "NLB not found. Check if the K8s Service was created and the LB controller is running."

print(f"\nNLB DNS: {NLB_DNS}")
print(f"NLB SG:  {NLB_SG_ID}")

# VPC CIDR에서 들어오는 443 포트 트래픽에 대해 NLB SG 열기
if NLB_SG_ID:
    try:
        ec2_client.authorize_security_group_ingress(
            GroupId=NLB_SG_ID,
            IpPermissions=[
                {
                    "IpProtocol": "tcp",
                    "FromPort": 443,
                    "ToPort": 443,
                    "IpRanges": [{"CidrIp": "10.0.0.0/16", "Description": "Allow TLS from VPC"}],
                }
            ],
        )
        print(f"Added inbound rule: {NLB_SG_ID} <- TCP 443 from 10.0.0.0/16")
    except ec2_client.exceptions.ClientError as e:
        if "InvalidPermission.Duplicate" in str(e):
            print(f"Inbound rule already exists on {NLB_SG_ID}")
        else:
            raise

# 프라이빗 호스팅 영역에서 NLB를 가리키는 Alias A 레코드 UPSERT
route53_client.change_resource_record_sets(
    HostedZoneId=PRIVATE_ZONE_ID,
    ChangeBatch={
        "Changes": [
            {
                "Action": "UPSERT",
                "ResourceRecordSet": {
                    "Name": PRIVATE_DOMAIN,
                    "Type": "A",
                    "AliasTarget": {
                        "HostedZoneId": NLB_HOSTED_ZONE_ID,
                        "DNSName": NLB_DNS,
                        "EvaluateTargetHealth": False,
                    },
                },
            }
        ]
    },
)
print(f"\nUPSERT-ed Alias record: {PRIVATE_DOMAIN} -> {NLB_DNS}")
print("Inside the VPC, the private domain now resolves to the NLB's private IPs.")

## 3단계: AgentCore Gateway 대상 생성(OpenAPI 스키마를 사용하는 MCP)

REST API에는 **OpenAPI 스키마**가 포함된 MCP 대상 유형을 사용합니다. AgentCore Gateway는 스키마를 사용하여 API 작업을 검색하고 AI 에이전트가 호출할 수 있는 도구로 노출합니다.

대상 엔드포인트는 프라이빗 FQDN입니다. Private DNS가 VPC 내부에서 이를 NLB의 프라이빗 IP로 해석하므로 `routingDomain`은 필요하지 않습니다.

> **보안 그룹:** Resource Gateway ENI가 포트 443을 통해 NLB에 도달할 수 있도록 NLB의 보안 그룹을 `securityGroupIds`에 전달합니다.

In [ ]:
# OpenAPI 스키마를 로드하고 서버 URL을 대상 엔드포인트로 설정
with open("05-eks-deployment/openapi.json") as f:
    openapi_schema = json.load(f)

openapi_schema["servers"] = [{"url": f"https://{DOMAIN}"}]

OPENAPI_SCHEMA = json.dumps(openapi_schema)
print(f"Loaded OpenAPI schema: {openapi_schema['info']['title']} v{openapi_schema['info']['version']}")
print(f"Server URL: {openapi_schema['servers'][0]['url']}")
print(f"Endpoints: {list(openapi_schema['paths'].keys())}")

In [ ]:
# 더미 API 키 자격 증명 공급자 생성(OpenAPI 대상에 필요)
# REST API는 인증을 사용하지 않지만 AgentCore Gateway에는 credentialProviderConfigurations가 필요함
cred_response = agentcore.create_api_key_credential_provider(
    name="eks-api-server-api-key",
    apiKey="dummy-key-for-eks-api",
)
CRED_PROVIDER_ARN = cred_response["credentialProviderArn"]
print(f"Credential provider ARN: {CRED_PROVIDER_ARN}")

In [ ]:
TARGET_ENDPOINT = f"https://{DOMAIN}"

print(f"Target endpoint: {TARGET_ENDPOINT}  (resolves via Private DNS inside the VPC)")

managed_vpc_resource_config = {
    "vpcIdentifier": VPC_USW2_ID,
    "subnetIds": VPC_USW2_PRIVATE_SUBNETS,
    "endpointIpAddressType": "IPV4",
}
if NLB_SG_ID:
    managed_vpc_resource_config["securityGroupIds"] = [NLB_SG_ID]

response = agentcore.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name="eks-api-server",
    description="REST API on EKS via internal NLB and managed VPC egress (Private DNS)",
    targetConfiguration={
        "mcp": {
            "openApiSchema": {
                "inlinePayload": OPENAPI_SCHEMA,
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "API_KEY",
            "credentialProvider": {
                "apiKeyCredentialProvider": {
                    "providerArn": CRED_PROVIDER_ARN,
                    "credentialParameterName": "x-api-key",
                    "credentialLocation": "HEADER",
                }
            },
        }
    ],
    privateEndpoint={
        "managedVpcResource": managed_vpc_resource_config,
    },
)

TARGET_ID = response["targetId"]
print(f"\nTarget ID: {TARGET_ID}")
print(f"Status:    {response['status']}")

In [ ]:
while True:
    target = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
    status = target["status"]
    print(f"Status: {status}")
    if status == "READY":
        print("\nTarget is active!")
        print(f"  Managed resources: {target.get('privateEndpoint', {})}")
        break
    if status == "FAILED":
        print(f"\nTarget creation failed: {target.get('statusReasons', [])}")
        break
    time.sleep(30)

## 4단계: AgentCore Gateway를 통해 API 호출

Cognito에서 액세스 토큰을 가져온 다음 Gateway를 통해 REST API 작업을 MCP 도구로 호출합니다.

In [ ]:
token_response = get_token(
    token_endpoint_url=TOKEN_ENDPOINT_URL,
    client_id=USER_POOL_CLIENT_ID,
    client_secret=CLIENT_SECRET,
    scope_string=OAUTH_SCOPES.replace(",", " "),
)
ACCESS_TOKEN = token_response["access_token"]
print(f"Access token obtained (expires in {token_response['expires_in']}s)")

In [ ]:
import requests

headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json",
}

# 사용 가능한 도구 목록 조회(MCP 도구로 노출된 API 작업)
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={"jsonrpc": "2.0", "method": "tools/list", "id": 1},
)
print("Available tools:")
print(json.dumps(response.json(), indent=2))

In [ ]:
# API를 통해 항목 생성(MCP 도구로 호출)
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {
            "name": "eks-api-server___createItem",
            "arguments": {"name": "Widget", "price": 9.99},
        },
        "id": 2,
    },
)
print("Created item:")
print(json.dumps(response.json(), indent=2))

In [ ]:
# API를 통해 항목 목록 조회
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {
            "name": "eks-api-server___listItems",
            "arguments": {},
        },
        "id": 3,
    },
)
print("Items:")
print(json.dumps(response.json(), indent=2))

## 정리

1. Gateway 대상 삭제
2. CDK 스택 삭제

> **참고:** [MCP 서버 Notebook](./mcp-server-gateway-managed.ipynb)을 실행하지 않는 경우에만 Shared EKS Cluster를 삭제하세요.

In [ ]:
# # 1단계: Gateway 대상 삭제
# agentcore.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
# print(f"Deleting target: {TARGET_ID}")
# while True:
#     try:
#         t = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
#         print(f"  Status: {t['status']}")
#         time.sleep(15)
#     except agentcore.exceptions.ResourceNotFoundException:
#         print("  Target deleted.")
#         break

# # 자격 증명 공급자 삭제
# agentcore.delete_api_key_credential_provider(name="eks-api-server-api-key")
# print("Deleted credential provider")

# # 프라이빗 호스팅 영역에서 Alias 레코드 삭제
# # (CDK는 레코드가 남아 있는 호스팅 영역을 삭제할 수 없음)
# route53_client.change_resource_record_sets(
#     HostedZoneId=PRIVATE_ZONE_ID,
#     ChangeBatch={
#         "Changes": [
#             {
#                 "Action": "DELETE",
#                 "ResourceRecordSet": {
#                     "Name": PRIVATE_DOMAIN,
#                     "Type": "A",
#                     "AliasTarget": {
#                         "HostedZoneId": NLB_HOSTED_ZONE_ID,
#                         "DNSName": NLB_DNS,
#                         "EvaluateTargetHealth": False,
#                     },
#                 },
#             }
#         ]
#     },
# )
# print(f"Deleted Alias record: {PRIVATE_DOMAIN} -> {NLB_DNS}")

In [ ]:
# # 2단계: CDK 스택 삭제
# !ACCOUNT_A_ID={ACCOUNT_A_ID} cdk destroy ApiEks \
#     -c "publicCertArn={CERT_ARN}" \
#     -c "privateDomain={DOMAIN}" \
#     --profile {ACCOUNT_A_PROFILE} --force

In [ ]:
# # mcp-server-gateway-managed.ipynb Notebook을 실행하지 않을 경우에만 EKS 클러스터 삭제
# !ACCOUNT_A_ID={ACCOUNT_A_ID} cdk destroy SharedEksCluster \
#     --profile {ACCOUNT_A_PROFILE} --force